In [ ]:
import os
import numpy as np
import pandas as pd
import logging
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
from tabpfn import TabPFNClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

PAPER_FIGSIZE_COL = (3.45, 2.35)
PAPER_FIGSIZE_WIDE = (7.16, 2.65)
PAPER_FIGSIZE_TALL = (3.45, 3.2)
PAPER_PLOT_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.title_fontsize": 7,
    "axes.linewidth": 0.8,
    "grid.linewidth": 0.35,
    "lines.linewidth": 1.35,
    "savefig.dpi": 600,
}
PAPER_COLOR_COUNT = "#2F6F73"
PAPER_COLOR_COMPARE = "#B85C38"
PAPER_COLOR_CLEAN = "#4F7F52"
PAPER_COLOR_ACCENT = "#8A6F2A"
PAPER_COLOR_LIGHT = "#F4F1E8"
PAPER_COLOR_INK = "#202020"
PAPER_CMAP = LinearSegmentedColormap.from_list("paper_count", [PAPER_COLOR_LIGHT, PAPER_COLOR_COUNT])
PAPER_FAMILY_COLORS = {
    "Clean_Label": PAPER_COLOR_COUNT,
    "Family_SVM": PAPER_COLOR_COMPARE,
    "Label_Flip": PAPER_COLOR_ACCENT,
    "clean": PAPER_COLOR_CLEAN,
}
METHOD_DISPLAY_NAMES = {
    "alfa_svm": "ALFA SVM",
    "art_svm": "ART SVM",
    "badnets": "BadNets",
    "clean": "Clean",
    "feature_collision": "Feature Collision",
    "feature_noise_svm": "Feature Noise SVM",
    "learning_to_confuse": "Learning to Confuse",
    "metapoison": "MetaPoison",
    "poison_frogs": "Poison Frogs",
    "poissvm_svm": "PoisSVM",
    "random_flip_svm": "Random Flip SVM",
    "witches_brew": "Witches' Brew",
}
METHOD_ORDER = [
    "clean",
    "random_flip_svm",
    "alfa_svm",
    "feature_noise_svm",
    "feature_collision",
    "poissvm_svm",
    "art_svm",
    "learning_to_confuse",
    "metapoison",
    "witches_brew",
    "poison_frogs",
    "badnets",
]
METHOD_DISPLAY_ORDER = [METHOD_DISPLAY_NAMES[m] for m in METHOD_ORDER if m in METHOD_DISPLAY_NAMES]
PAPER_MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
PAPER_LINESTYLES = ["-", "--", "-.", ":"]
sns.set_theme(style="whitegrid", context="paper", rc=PAPER_PLOT_RC)
plt.rcParams.update(PAPER_PLOT_RC)

# Setup Logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s [%(name)s] [%(levelname)s] %(message)s', 
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("LOAO_Unified_Benchmark")

In [ ]:
def extract_base_dataset_name(dataname):
    if "_vs_" in dataname:
        part1 = dataname.split("_vs_")[0]
        return part1.rsplit("_", 1)[0]
    return dataname

In [ ]:
def prepare_benchmark_data(db_path, seed, split_target='Is_Poisoned'):
    """
    Standardized data loading, filtering, and train-test splitting 
    (mimicking the original classification benchmark approach).
    """
    df = pd.read_csv(db_path)
    df['BaseGroup'] = df['Data'].apply(extract_base_dataset_name)

    # feature_noise_svm is considered clean
    df.loc[df['Method'] == 'feature_noise_svm', 'Is_Poisoned'] = 0
    df.loc[df['Method'] == 'feature_noise_svm', 'Method'] = 'clean'
    
    drop_cols = ['Data', 'Path', 'Method', 'Rate', 'Is_Poisoned', 'error', 'BaseGroup']
    drop_cols += [c for c in df.columns if c in ['Train.Clean', 'Test.Clean', 'Train.Poison', 'Test.Poison']]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    
    logger.info(f"Using all {len(feature_cols)} available features.")

    df[feature_cols] = df[feature_cols].fillna(0)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(df[feature_cols], df[split_target], df['BaseGroup']))
    
    train_df_full = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    # Leave out specific poisoners as defined in classification
    poisoner_to_leave_out = ["diva_attack_xgb", "diva_attack", "diva_attack2"]
    train_df_full = train_df_full[~(train_df_full['Method'].isin(poisoner_to_leave_out))]
    test_df = test_df[~(test_df['Method'].isin(poisoner_to_leave_out))]

    # Identify methods
    poisoners = sorted([m for m in train_df_full['Method'].unique() if m != 'clean'])
    all_methods = sorted(list(train_df_full['Method'].unique()))
    
    logger.info(f"Identified {len(poisoners)} poisoners to test: {poisoners}")
    
    return train_df_full, test_df, feature_cols, poisoners, all_methods

In [ ]:
db_path = "meta_db_universal.csv"
workers = 4
seed = 42
binary_model_path = "data/binary_detector.joblib"
plots_dir = f"data/plots_multiclass"
os.makedirs(plots_dir, exist_ok=True)

hard_ood_threshold = 0.60
binary_threshold = 0.76

In [ ]:
logger.info(f"--- Starting Two-Stage Cascading Benchmark ---")

if not os.path.exists(binary_model_path):
    logger.error(f"Binary model not found at {binary_model_path}. Please provide a valid path.")

logger.info(f"Loading Binary Detector from {binary_model_path}")
binary_clf = joblib.load(binary_model_path)


# 1. Load Data
train_df_full, test_df, feature_cols, poisoners, all_methods = prepare_benchmark_data(
    db_path, seed, split_target='Method'
)

poisoners = sorted([m for m in train_df_full['Method'].unique() if m != 'clean'])

# 2. Define and Apply Family Mapping
family_mapping = {
    'art_svm': 'Family_SVM',
    'poissvm_svm': 'Family_SVM',
    
    'learning_to_confuse': 'Clean_Label',
    'badnets': 'Clean_Label',
    'metapoison': 'Clean_Label',
    'witches_brew': 'Clean_Label',
    'feature_collision': 'Clean_Label',
    'poison_frogs': 'Clean_Label',
    
    'alfa_svm': 'Label_Flip',
    'random_flip_svm': 'Label_Flip',
    
    'clean': 'clean'
}

train_df_full['Family'] = train_df_full['Method'].map(family_mapping)
test_df['Family'] = test_df['Method'].map(family_mapping)

In [ ]:
# TabPFN multiclass baseline training disabled for plotting-only reruns. Uncomment only to regenerate caches.
# # =========================================================================
# # 3. Full Dataset Evaluation (Baseline - No Holdouts)
# # =========================================================================
# logger.info(f"🚀 Training Family Classifier: [BASELINE ALL DATA - NO HOLDOUTS]")
#
# train_df_poison = train_df_full[train_df_full['Family'] != 'clean'].copy()
# le_all = LabelEncoder()
# y_train_all = le_all.fit_transform(train_df_poison['Family'])
#
# clf_multi_all = TabPFNClassifier(n_estimators=32, device='auto', random_state=seed)
# clf_multi_all.fit(train_df_poison[feature_cols], y_train_all)
#
# X_test_all = test_df[feature_cols]
#
# # Stage 1: Binary Prediction
# bin_probs_all = binary_clf.predict_proba(X_test_all)[:, 1]
# bin_preds_all = (bin_probs_all >= binary_threshold).astype(int)
#
# # Stage 2: Multiclass Prediction & OOD
# multi_probs_all = clf_multi_all.predict_proba(X_test_all)
# multi_preds_all = le_all.inverse_transform(np.argmax(multi_probs_all, axis=1))

In [ ]:
# Load cached multiclass baseline results for plotting-only reruns.
multiclass_baseline_cache = os.path.join(plots_dir, "multiclass_baseline_predictions.csv")
if not os.path.exists(multiclass_baseline_cache):
    raise FileNotFoundError(
        "Cached multiclass baseline results are missing. Required for plotting without TabPFN retraining: "
        + multiclass_baseline_cache
    )

test_df_all = pd.read_csv(multiclass_baseline_cache)
test_df = test_df_all.copy()
max_probs = test_df_all["Max_Multi_Prob"].to_numpy()
bin_preds_all = test_df_all["Binary_Prediction"].to_numpy()
multi_preds_all = test_df_all["Multiclass_Prediction"].to_numpy()
multi_probs_all = test_df_all[[c for c in test_df_all.columns if c.startswith("Prob_")]].to_numpy()
logger.info("Loaded cached multiclass baseline results for plotting.")


In [ ]:
"""
Plots the rate of datasets classified as OOD vs. the Maximum Probability Threshold,
broken down by the true Family (including clean).
"""
# Calculate the maximum predicted probability for each test instance
max_probs = np.max(multi_probs_all, axis=1)

# Define a range of thresholds to test (from 0 to 1 in 0.01 increments)
thresholds = np.linspace(0.0, 1.0, 101)

# Get unique families in the test set
families = sorted([f for f in test_df['Family'].unique() if pd.notna(f)])

plt.figure(figsize=PAPER_FIGSIZE_WIDE)
sns.set_theme(style="whitegrid", context="paper", rc=PAPER_PLOT_RC)

# Define a color palette
palette = [PAPER_FAMILY_COLORS.get(family, PAPER_COLOR_COUNT) for family in families]

for idx, family in enumerate(families):
    # Isolate the probabilities for just this family
    family_mask = test_df['Family'] == family
    family_max_probs = max_probs[family_mask]
    
    if len(family_max_probs) == 0:
        continue
        
    ood_rates = []
    for t in thresholds:
        # If the max probability is strictly less than the threshold, it is flagged as OOD
        rate = (family_max_probs < t).mean()
        ood_rates.append(rate)
        
    # Highlight 'clean' with a dashed line to differentiate it from attacks
    lw = 1.6 if family == 'clean' else 1.2
    ls = '--' if family == 'clean' else PAPER_LINESTYLES[idx % len(PAPER_LINESTYLES)]
    marker = PAPER_MARKERS[idx % len(PAPER_MARKERS)]
    
    plt.plot(thresholds, ood_rates, label=family, color=palette[idx], linewidth=lw, linestyle=ls, marker=marker, markevery=10, markersize=3)
    
plt.xlabel("Maximum Probability Threshold")
plt.ylabel("OOD Rate")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.05)

# Move legend outside the plot
plt.legend(title="True Family", bbox_to_anchor=(1.01, 1), loc='upper left', frameon=True)
plt.tight_layout()

plot_path = os.path.join(plots_dir, "ood_rate_vs_threshold_per_family.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    
logger.info(f"   => Saved OOD Threshold Curve to {plot_path}")

In [ ]:
is_ood_all = np.max(multi_probs_all, axis=1) < hard_ood_threshold

test_df_all = test_df.copy()
test_df_all['Pipeline_Prediction'] = np.where(
    bin_preds_all == 0, 
    'clean', 
    np.where(is_ood_all, 'Out_of_Distribution', multi_preds_all)
)

all_data_accs = []
logger.info("   => [ALL DATA] Pipeline Accuracies per Family:")
for family in test_df_all['Family'].unique():
    mask = test_df_all['Family'] == family
    if mask.sum() > 0:
        acc = accuracy_score(test_df_all.loc[mask, 'Family'], test_df_all.loc[mask, 'Pipeline_Prediction'])
        all_data_accs.append({'Family': family, 'Accuracy': acc})
        logger.info(f"      - {family}: {acc:.2%}")
        
df_all_accs = pd.DataFrame(all_data_accs)

# Standard Heatmap Confusion Matrix (Baseline)
# Using crosstab allows us to have asymmetric rows and columns automatically
cm_df_all = pd.crosstab(
    test_df_all['Family'], 
    test_df_all['Pipeline_Prediction'], 
    rownames=['True Family'], 
    colnames=['Predicted Pipeline Family'],
    normalize='index' # Normalizes over rows directly
).fillna(0)

# Ensure standard ordering
expected_true_all = sorted(test_df_all['Family'].unique())
expected_pred_all = expected_true_all + ['Out_of_Distribution']
cm_df_all = cm_df_all.reindex(index=expected_true_all, columns=expected_pred_all, fill_value=0.0)

fig_cm_all, ax_cm_all = plt.subplots(figsize=(7.16, 3.35))
sns.heatmap(cm_df_all, annot=True, fmt='.2f', cmap=PAPER_CMAP, vmin=0, vmax=1, ax=ax_cm_all, linewidths=0.35, linecolor='white', cbar_kws={'label': 'Proportion'})
plt.setp(ax_cm_all.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
fig_cm_all.savefig(os.path.join(plots_dir, "cascade_cm_all_data.png"), dpi=300, bbox_inches='tight')

In [ ]:
# TabPFN zero-day holdout loop disabled for plotting-only reruns. Uncomment only to regenerate caches.
# # =========================================================================
# # 4. Zero-Day Generalization & OOD Loop
# # =========================================================================
# ood_results = []
# all_test_preds = [] 
#
# for holdout in poisoners:
#     logger.info(f"🚀 Training Cascade Multiclassifier: [HOLDOUT METHOD: {holdout.upper()}]")
#
#     train_df = train_df_full[(train_df_full['Method'] != holdout) & (train_df_full['Family'] != 'clean')].copy()
#     le = LabelEncoder()
#     y_train_encoded = le.fit_transform(train_df['Family'])
#     X_train = train_df[feature_cols]
#     X_test = test_df[feature_cols]
#
#     clf_multi = TabPFNClassifier(n_estimators=32, device='auto', random_state=seed)
#     clf_multi.fit(X_train, y_train_encoded)
#
#     bin_probs = binary_clf.predict_proba(X_test)[:, 1]
#     bin_preds = (bin_probs >= binary_threshold).astype(int)
#
#     multi_probs = clf_multi.predict_proba(X_test)
#     multi_preds = le.inverse_transform(np.argmax(multi_probs, axis=1))
#     is_ood = np.max(multi_probs, axis=1) < hard_ood_threshold
#
#     test_df_copy = test_df.copy()
#     test_df_copy['Is_Zero_Day'] = (test_df_copy['Method'] == holdout).astype(int)
#     test_df_copy['Holdout'] = holdout
#
#     pipeline_preds = np.where(
#         bin_preds == 0, 
#         'clean', 
#         np.where(is_ood, 'Out_of_Distribution', multi_preds)
#     )
#     test_df_copy['Pipeline_Prediction'] = pipeline_preds
#
#     all_test_preds.append(test_df_copy)
#
#     zero_day_df = test_df_copy[test_df_copy['Is_Zero_Day'] == 1]
#
#     known_poison_mask = (test_df_copy['Is_Zero_Day'] == 0) & (test_df_copy['Family'] != 'clean')
#     known_acc = accuracy_score(
#         test_df_copy.loc[known_poison_mask, 'Family'], 
#         test_df_copy.loc[known_poison_mask, 'Pipeline_Prediction']
#     ) if known_poison_mask.sum() > 0 else 0
#
#     zero_day_family_acc = (zero_day_df['Pipeline_Prediction'] == zero_day_df['Family']).mean() if not zero_day_df.empty else 0
#     zero_day_ood_rate = (zero_day_df['Pipeline_Prediction'] == 'Out_of_Distribution').mean() if not zero_day_df.empty else 0
#
#     clean_mask = test_df_copy['Family'] == 'clean'
#     clean_acc = accuracy_score(
#         test_df_copy.loc[clean_mask, 'Family'], 
#         test_df_copy.loc[clean_mask, 'Pipeline_Prediction']
#     ) if clean_mask.sum() > 0 else 0
#
#     ood_results.append({
#         'Holdout_Poisoner': holdout,
#         'True_Macro_Family': family_mapping.get(holdout, 'Unknown'),
#         'Known_Poison_Family_Acc': known_acc,
#         'Zero_Day_Intra_Family_Acc': zero_day_family_acc,
#         'Zero_Day_OOD_Rate': zero_day_ood_rate,
#         'Clean_Retention_Acc': clean_acc
#     })

In [ ]:
# Load cached zero-day table results for plotting-only reruns.
csv_path = os.path.join(plots_dir, "holdout_accuracies_table.csv")
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Cached zero-day table is missing. Required for table rendering without TabPFN retraining: "
        + csv_path
    )
paper_table_df = pd.read_csv(csv_path)
logger.info("Loaded cached zero-day accuracy table for rendering.")


In [ ]:
# Render cached accuracy table as a paper-sized image.
fig_tbl, ax_tbl = plt.subplots(figsize=(7.16, min(4.8, max(2.2, 0.34 * (len(paper_table_df) + 1)))))
ax_tbl.axis('off')
ax_tbl.axis('tight')
tbl = ax_tbl.table(
    cellText=paper_table_df.values, 
    colLabels=paper_table_df.columns, 
    cellLoc='center', 
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(0.92, 1.25)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(PAPER_COLOR_INK)
    cell.set_linewidth(0.4)
    if row == 0:
        cell.set_facecolor(PAPER_COLOR_LIGHT)
        cell.set_text_props(weight='bold')

tbl_path = os.path.join(plots_dir, "holdout_accuracies_table.png")
fig_tbl.savefig(tbl_path, dpi=300, bbox_inches='tight')
